# MiniMax H3 — Director + Refiner + Episode Assembly
Use a G4 / RTX PRO 6000 Blackwell runtime. Upload the accompanying `Logan_H3_Director_Combined_Pipeline.zip` below. This launcher runs the matching standalone notebook from that release instead of mixing it with the older Extender scripts.
The source bundle contains the implementation; model weights are downloaded by its setup. Reference pictures and voice recordings belong in your Colab/ComfyUI project, not in this public repository.
For editing setup options cell by cell, open the standalone `Logan_H3_Director_Combined_Colab.ipynb` supplied with the ZIP directly in Colab instead.
GPU execution remains a live-runtime validation step. Do not run a full episode until one clip, its reference voice, refinement, and export have succeeded.


## 1. Upload and validate the release bundle
Select only the pipeline ZIP downloaded from this conversation. This unpacks into a new temporary directory without deleting existing ComfyUI files.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, io, json, stat, tempfile, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Select only Logan_H3_Director_Combined_Pipeline.zip.')
payload = next(iter(uploaded.values()))
release_dir = Path(tempfile.mkdtemp(prefix='logan_h3_release_', dir='/content'))
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    if sum(item.file_size for item in archive.infolist()) > 100 * 1024**2:
        raise RuntimeError('The source bundle must not contain model weights or large media.')
    for item in archive.infolist():
        relative = PurePosixPath(item.filename)
        if relative.is_absolute() or '..' in relative.parts or '\\' in item.filename:
            raise RuntimeError('Unsafe archive path: ' + item.filename)
        if stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Symlinks are not accepted in the source bundle.')
        target = release_dir.joinpath(*relative.parts)
        if item.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(item))
status = json.loads((release_dir / 'DELIVERY_STATUS.json').read_text())
for relative, details in status.get('files', {}).items():
    local = release_dir / 'minimax_h3_director' / relative
    if not local.is_file() or hashlib.sha256(local.read_bytes()).hexdigest() != details['sha256']:
        raise RuntimeError('Source checksum mismatch: ' + relative)
notebook_path = release_dir / 'Logan_H3_Director_Combined_Colab.ipynb'
release_notebook = json.loads(notebook_path.read_text())
if release_notebook.get('nbformat') != 4 or not release_notebook.get('cells'):
    raise RuntimeError('The matching standalone notebook is missing or invalid.')
print('Release source validated:', release_dir)
print('Source files:', len(status.get('files', {})))


## 2. Run the matching setup
This executes the release notebook's cells in order and stops on any error. Follow the Drive, GPU, model-download and reference-upload prompts it displays. A missing GPU or dependency is not silently treated as success.


In [ ]:
from IPython.display import display, Markdown
shell = get_ipython()
for index, cell in enumerate(release_notebook['cells'], 1):
    source = cell.get('source', '')
    source = ''.join(source) if isinstance(source, list) else source
    if cell.get('cell_type') == 'markdown':
        display(Markdown(source))
    elif cell.get('cell_type') == 'code' and source.strip():
        print(f'Running release cell {index}/{len(release_notebook["cells"])}')
        result = shell.run_cell(source, store_history=False)
        error = result.error_before_exec or result.error_in_exec
        if error is not None:
            raise RuntimeError(f'Release setup stopped at cell {index}: {error}') from error
print('Release notebook cells finished. Verify one generated clip before a long run.')
